# HotelRec – Group Recommender (End-to-End)

This notebook builds a content-based recommendation engine for individual users and extends it with group-level recommendations and evaluation.


In [10]:
import os, glob, ast, json
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [11]:
# --- Data loading ---
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), '.', '.'))

# EITHER a single file, OR a directory
DATA_PATH = r"C:\Users\yannk\OneDrive\Dokumente\hotelrec_dense_k20_Yann.csv"
LIMIT_PER_FILE = None
MIN_USER = 3
MIN_ITEM = 3

paths = []

# Case 1: DATA_PATH is a single CSV file
if os.path.isfile(DATA_PATH) and DATA_PATH.lower().endswith(".csv"):
    paths = [DATA_PATH]
    print(f"Using single CSV file: {DATA_PATH}")

# Case 2: DATA_PATH is a directory containing many CSVs
elif os.path.isdir(DATA_PATH):
    paths = sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))
    print(f"Using data directory: {DATA_PATH}")

# Case 3: relative path or something else → try resolving under PROJECT_ROOT/data*/filtered
else:
    candidate_dirs = [
        os.path.join(PROJECT_ROOT, DATA_PATH),
        os.path.join(PROJECT_ROOT, "data", "filtered"),
        os.path.join(PROJECT_ROOT, "data"),
    ]
    for base in candidate_dirs:
        if os.path.isdir(base):
            shard_paths = sorted(glob.glob(os.path.join(base, "*.csv")))
            if shard_paths:
                paths = shard_paths
                DATA_PATH = base
                print(f"Using data directory: {DATA_PATH}")
                break

if not paths:
    raise FileNotFoundError(
        f"Could not find any CSV files. Checked:\n"
        f"  - file: {DATA_PATH}\n"
        f"  - and candidate directories under project root.\n"
        f"Please adjust DATA_PATH."
    )

print(f"Found {len(paths)} CSV shard(s).")

dfs = []
for p in paths:
    print("Reading", os.path.basename(p))
    df = pd.read_csv(p)
    if LIMIT_PER_FILE is not None:
        df = df.head(LIMIT_PER_FILE)
    dfs.append(df)

ratings = pd.concat(dfs, ignore_index=True)

# Basic typing & filtering
# --- Map columns from your CSV to canonical names: user, item, rating ---

print("Columns in loaded CSV:", ratings.columns.tolist())

# Try to guess column names
user_candidates   = ["user", "user_id", "userid", "userId", "reviewer", "customer_id"]
item_candidates   = ["item", "item_id", "itemid", "itemId", "hotel_id", "sid", "s_id"]
rating_candidates = ["rating", "score", "stars", "overall", "rating_val"]

def pick_col(candidates, what):
    for c in candidates:
        if c in ratings.columns:
            print(f"Using column '{c}' as {what}")
            return c
    raise KeyError(
        f"Could not find a column for {what}. "
        f"Tried: {candidates}. "
        f"Available columns: {ratings.columns.tolist()}"
    )

user_col   = pick_col(user_candidates,   "user")
item_col   = pick_col(item_candidates,   "item")
rating_col = pick_col(rating_candidates, "rating")

# Rename to canonical names
ratings = ratings.rename(columns={
    user_col: "user",
    item_col: "item",
    rating_col: "rating",
})
ratings["user"] = ratings["user"].astype(str)
ratings["item"] = ratings["item"].astype(str)
ratings["rating"] = ratings["rating"].astype(np.float32)

user_counts = ratings["user"].value_counts()
item_counts = ratings["item"].value_counts()
ratings = ratings[
    ratings["user"].isin(user_counts.index[user_counts >= MIN_USER]) &
    ratings["item"].isin(item_counts.index[item_counts >= MIN_ITEM])
].reset_index(drop=True)

print(f"Loaded ratings: {len(ratings):,} rows")
print(f"Users: {ratings['user'].nunique():,}, Items: {ratings['item'].nunique():,}")



Using single CSV file: C:\Users\yannk\OneDrive\Dokumente\hotelrec_dense_k20_Yann.csv
Found 1 CSV shard(s).
Reading hotelrec_dense_k20_Yann.csv
Columns in loaded CSV: ['user_id', 'item_id', 'rating', 'sleep quality', 'value', 'rooms', 'service', 'cleanliness', 'location', 'text']
Using column 'user_id' as user
Using column 'item_id' as item
Using column 'rating' as rating
Loaded ratings: 1,400,102 rows
Users: 46,660, Items: 27,197


In [12]:
# --- Train/test split: per-user holdout ---
def make_per_user_holdout(ratings: pd.DataFrame, test_frac: float = 0.2, seed: int = 42):
    rng = np.random.default_rng(seed)
    df = ratings.sample(frac=1.0, random_state=seed)
    train_parts = []
    test_dict: Dict[str, set] = {}
    for u, g in df.groupby('user', observed=False):
        items = g['item'].values
        if len(items) < 2:
            train_parts.append(g)
            continue
        n_test = max(1, int(np.floor(test_frac * len(items))))
        idx = rng.choice(len(items), size=n_test, replace=False)
        test_items = set(items[idx].astype(str).tolist())
        train_parts.append(g.drop(index=g.index[idx]))
        test_dict[u] = test_items
    train_df = pd.concat(train_parts, ignore_index=True)
    return train_df, test_dict

train_df, test_dict = make_per_user_holdout(ratings, test_frac=0.2, seed=RANDOM_SEED)
print(f'Train interactions: {len(train_df):,}')
print(f'Users with test data: {len(test_dict):,}')


Train interactions: 1,136,467
Users with test data: 46,660


In [ ]:
# %%
# --- Core interfaces and utilities ---
from abc import ABC, abstractmethod

class BaseAlgo(ABC):
    @abstractmethod
    def fit(self, ratings: pd.DataFrame):
        pass

    @abstractmethod
    def predict_for_user(self, user: str, candidates=None, n: int = 10) -> pd.DataFrame:
        pass

    def recommend(self, user: str, n: int = 10, candidates=None) -> pd.DataFrame:
        res = self.predict_for_user(user, candidates=candidates, n=n)
        return res.head(n) if isinstance(res, pd.DataFrame) else res


def build_user_hist(ratings: pd.DataFrame) -> Dict[str, set]:
    return ratings.groupby('user', observed=False)['item'].apply(set).to_dict()


def filter_seen(recs: pd.DataFrame, seen_items):
    if recs is None or recs.empty or not seen_items:
        return recs
    return recs[~recs['item'].isin(seen_items)].reset_index(drop=True)


class RecEngine:
    """
    Wrapper around a primary algorithm (e.g. TF-IDF) and optional fallback (e.g. popularity).
    Handles:
      - user histories
      - candidate generation
      - filtering seen items
    """
    def __init__(self, algo: BaseAlgo, fallback: Optional[BaseAlgo] = None):
        self.algo = algo
        self.fallback = fallback
        self.user_hist: Dict[str, set] = {}
        self.all_items_: List[str] = []

    def fit(self, ratings: pd.DataFrame):
        # Store per-user history and item catalog from the *training* data
        self.user_hist = build_user_hist(ratings)
        self.all_items_ = sorted(ratings['item'].astype(str).unique())

        # Fit primary and fallback models
        self.algo.fit(ratings)
        if self.fallback is not None:
            self.fallback.fit(ratings)
        return self

    def recommend(
        self,
        user: str,
        n: int = 10,
        candidates=None,
        filter_seen_items: bool = True
    ) -> pd.DataFrame:
        """
        Recommend top-n items for a single user.
        """
        seen = self.user_hist.get(user, set())

        # If no candidate list is given, use all items minus seen ones.
        if candidates is None:
            candidates = [it for it in self.all_items_ if it not in seen]
        else:
            candidates = [str(c) for c in candidates]

        # Try primary algo
        try:
            recs = self.algo.predict_for_user(user, candidates=candidates, n=n)
        except Exception:
            recs = None

        # Fallback if primary fails or returns nothing
        if (recs is None or recs.empty) and self.fallback is not None:
            recs = self.fallback.predict_for_user(user, candidates=candidates, n=n)

        if recs is None:
            return pd.DataFrame(columns=['item', 'score'])

        if filter_seen_items:
            recs = filter_seen(recs, seen)

        return recs.head(n).reset_index(drop=True)


class PopularityAlgo(BaseAlgo):
    def __init__(self, mode: str = 'count', min_count: int = 1):
        """
        mode: 'count' = rank by rating count
              'mean'  = rank by average rating
        """
        self.mode = mode
        self.min_count = min_count
        self.item_scores_: Optional[pd.DataFrame] = None

    def fit(self, ratings: pd.DataFrame):
        grp = ratings.groupby('item')['rating']
        counts = grp.size()
        means = grp.mean()

        df = pd.DataFrame({
            'item': counts.index.astype(str),
            'count': counts.values,
            'mean': means.values
        })

        df = df[df['count'] >= self.min_count].copy()
        if self.mode == 'mean':
            df['score'] = df['mean']
        else:
            df['score'] = df['count']

        df = df.sort_values('score', ascending=False).reset_index(drop=True)
        self.item_scores_ = df[['item', 'score']]
        return self

    def predict_for_user(self, user: str, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_scores_ is None:
            raise RuntimeError('PopularityAlgo must be fitted before prediction.')
        df = self.item_scores_.copy()
        if candidates is not None:
            cand_set = set(str(c) for c in candidates)
            df = df[df['item'].isin(cand_set)]
        return df.head(n).reset_index(drop=True)

print('✓ Core interfaces (BaseAlgo, RecEngine, PopularityAlgo) ready.')


✓ Core interfaces (BaseAlgo, RecEngine, PopularityAlgo) ready.


In [ ]:
# --- TF-IDF content-based algorithm (simplified) ---
class TFIDFContentAlgo(BaseAlgo):
    def __init__(self, min_df: int = 3, max_df: float = 0.7, max_features: int = 70000,
                 ngram_range: Tuple[int, int] = (1, 2), like_threshold: float = 3.0):
        self.min_df = min_df
        self.max_df = max_df
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.like_threshold = like_threshold

        self.vectorizer: Optional[TfidfVectorizer] = None
        self.item_ids_: List[str] = []
        self.item_matrix_: Optional[sp.csr_matrix] = None
        self.user_item_ratings_: Dict[str, List[Tuple[str, float]]] = {}

    def _build_item_texts(self, ratings: pd.DataFrame) -> pd.DataFrame:
        if 'text' not in ratings.columns:
            raise ValueError("Ratings dataframe must contain a 'text' column with review text.")
        df = ratings[['item', 'text']].dropna()
        agg = df.groupby('item', observed=False)['text'].apply(lambda x: ' '.join(map(str, x))).reset_index()
        return agg

    def fit(self, ratings: pd.DataFrame):
        # Build item texts
        items_df = self._build_item_texts(ratings)
        self.item_ids_ = items_df['item'].astype(str).tolist()

        # TF-IDF matrix for items
        self.vectorizer = TfidfVectorizer(
            min_df=self.min_df,
            max_df=self.max_df,
            max_features=self.max_features,
            ngram_range=self.ngram_range,
            stop_words='english'
        )
        X = self.vectorizer.fit_transform(items_df['text'].astype(str))
        self.item_matrix_ = sk_normalize(X, norm='l2', axis=1)

        # Build per-user interaction list WITHOUT groupby.apply
        self.user_item_ratings_ = {}
        for u, g in ratings.groupby('user', observed=False):
            items = g['item'].astype(str).tolist()
            ratings_u = g['rating'].astype(float).tolist()
            self.user_item_ratings_[u] = list(zip(items, ratings_u))
        return self


    def _build_user_profile(self, user: str):
        if self.item_matrix_ is None or not self.user_item_ratings_:
            raise RuntimeError('Model is not fitted.')
        interactions = self.user_item_ratings_.get(user)
        if not interactions:
            return None

        liked = [(iid, r) for iid, r in interactions if r >= self.like_threshold]
        if not liked:
            liked = interactions

        item_index = {iid: idx for idx, iid in enumerate(self.item_ids_)}
        rows = []
        weights = []
        for iid, r in liked:
            idx = item_index.get(iid)
            if idx is None:
                continue
            rows.append(self.item_matrix_[idx])
            weights.append(float(r))
        if not rows:
            return None
        mat = sp.vstack(rows)
        weights = np.array(weights).reshape(-1, 1)
        prof = (mat.multiply(weights)).sum(axis=0)
        prof = np.asarray(prof).ravel()
        norm = np.linalg.norm(prof)
        if norm > 0:
            prof = prof / norm
        return prof

    def predict_for_user(self, user: str, candidates=None, n: int = 10) -> pd.DataFrame:
        prof = self._build_user_profile(user)
        if prof is None:
            return pd.DataFrame(columns=['item', 'score'])
        item_index = {iid: idx for idx, iid in enumerate(self.item_ids_)}
        if candidates is None:
            cand_ids = self.item_ids_
        else:
            cand_ids = [str(c) for c in candidates if str(c) in item_index]
        if not cand_ids:
            return pd.DataFrame(columns=['item', 'score'])
        idxs = [item_index[iid] for iid in cand_ids]
        submat = self.item_matrix_[idxs]
        scores = submat.dot(prof)
        scores = np.asarray(scores).ravel()
        out = pd.DataFrame({'item': cand_ids, 'score': scores})
        out = out.sort_values('score', ascending=False).reset_index(drop=True)
        return out.head(n)

print('✓ TFIDFContentAlgo defined.')


✓ TFIDFContentAlgo defined.


In [15]:
# --- Train individual recommendation engine ---
content_algo = TFIDFContentAlgo(
    min_df=2,
    max_df=0.77,
    max_features=70000,
    ngram_range=(1, 2),
    like_threshold=3.2,
)

pop_fallback = PopularityAlgo(mode='count', min_count=5)

engine = RecEngine(algo=content_algo, fallback=pop_fallback).fit(train_df)
print('✓ Engine trained and ready for individual recommendations.')


C:\Users\yannk\AppData\Local\Temp\ipykernel_44184\2567946958.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g['item'].astype(str).tolist(), g['rating'].astype(float).tolist())))


✓ Engine trained and ready for individual recommendations.


In [16]:
# %%
# --- Individual-level evaluation (works also for group engines) ---
def eval_at_k_extended(
    engine,
    test_dict: Dict[str, set],
    k: int = 10,
    show_progress: bool = True,
    sample_size: Optional[int] = None,
) -> Dict[str, float]:
    """
    Generic evaluation for any object that has a .recommend(user, n=K) method.

    Metrics:
      - HitRate@K  : fraction of entities (users or groups) with ≥1 hit in top-K
      - Precision@K: macro-average over entities
      - Recall@K   : macro-average over entities
      - F1@K       : harmonic mean of macro Precision and macro Recall
      - NDCG@K     : average NDCG@K over entities
      - MRR@K      : mean reciprocal rank over entities
    """
    users = list(test_dict.keys())

    # Optional subsampling for speed
    if sample_size is not None and sample_size < len(users):
        rng = np.random.default_rng(RANDOM_SEED)
        users = rng.choice(users, size=sample_size, replace=False).tolist()

    hits = 0
    precisions = []
    recalls = []
    ndcgs = []
    rr_scores = []

    iterator = tqdm(users, desc=f'Evaluating @K={k}', leave=False) if show_progress else users

    for u in iterator:
        true_items = test_dict[u]
        if not true_items:
            continue

        rec_df = engine.recommend(u, n=k)
        rec_items = rec_df['item'].astype(str).tolist() if rec_df is not None and not rec_df.empty else []
        if not rec_items:
            continue

        true_set = set(str(x) for x in true_items)
        hits_in_topk = [int(i in true_set) for i in rec_items]
        n_rel = sum(hits_in_topk)

        # HitRate contribution
        if n_rel > 0:
            hits += 1

        # Per-entity precision & recall
        prec = n_rel / len(rec_items)
        rec = n_rel / len(true_set)

        precisions.append(prec)
        recalls.append(rec)

        # DCG + RR
        dcg = 0.0
        rr = 0.0
        for rank, is_rel in enumerate(hits_in_topk):
            if is_rel:
                dcg += 1.0 / np.log2(rank + 2)  # log2(rank+2) because rank starts at 0
                if rr == 0.0:
                    rr = 1.0 / (rank + 1)
        idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(true_set), k)))
        ndcg_val = (dcg / idcg) if idcg > 0 else 0.0

        ndcgs.append(ndcg_val)
        rr_scores.append(rr)

    num_users = len(users) if len(users) > 0 else 1   # FIXED

    hitrate = hits / num_users
    precision = float(np.mean(precisions)) if precisions else 0.0
    recall = float(np.mean(recalls)) if recalls else 0.0
    # F1 based on macro precision & recall (more stable / less "weird")
    if precision + recall > 0:
        f1 = 2.0 * precision * recall / (precision + recall)
    else:
        f1 = 0.0
    ndcg_mean = float(np.mean(ndcgs)) if ndcgs else 0.0
    mrr = float(np.mean(rr_scores)) if rr_scores else 0.0

    return {
        'HitRate@K': round(hitrate, 4),
        'Precision@K': round(precision, 4),
        'Recall@K': round(recall, 4),
        'F1@K': round(f1, 4),
        'NDCG@K': round(ndcg_mean, 4),
        'MRR@K': round(mrr, 4),
    }


# Quick sanity check: individual recommender quality
for k_val in [5, 10]:
    metrics = eval_at_k_extended(engine, test_dict, k=k_val, sample_size=5000, show_progress=True)
    print(f'[INDIVIDUAL] K={k_val}: {metrics}')


[INDIVIDUAL] K=5: {'HitRate@K': 0.0586, 'Precision@K': 0.0122, 'Recall@K': 0.0115, 'F1@K': 0.0118, 'NDCG@K': 0.0144, 'MRR@K': 0.0324}


[INDIVIDUAL] K=10: {'HitRate@K': 0.0942, 'Precision@K': 0.0103, 'Recall@K': 0.0193, 'F1@K': 0.0134, 'NDCG@K': 0.0174, 'MRR@K': 0.0371}


In [17]:
# %%
# --- Group recommendation: aggregation strategies ---
def aggregate_member_scores(
    member_frames: Dict[str, pd.DataFrame],
    strategy: str = 'average'
) -> pd.DataFrame:
    """
    Aggregate per-member recommendation scores into a single group ranking.

    member_frames: dict user_id -> df with columns ['item', 'score'].
    strategy:
        - 'average'       : mean over members (classic group-utility)
        - 'least_misery'  : min over members (penalize items disliked by anyone)
        - 'most_pleasure' : max over members (reward items liked by someone)
    """
    # Collect all candidate items
    all_items = set()
    for df in member_frames.values():
        if df is not None and not df.empty:
            all_items.update(df['item'].astype(str).tolist())
    if not all_items:
        return pd.DataFrame(columns=['item', 'score'])
    all_items = sorted(all_items)

    users = list(member_frames.keys())
    scores = np.zeros((len(all_items), len(users)), dtype=np.float32)
    item_to_idx = {it: i for i, it in enumerate(all_items)}

    # Fill score matrix
    for col, u in enumerate(users):
        df = member_frames[u]
        if df is None or df.empty:
            continue
        for _, row in df.iterrows():
            i = item_to_idx[str(row['item'])]
            scores[i, col] = float(row['score'])

    if strategy == 'average':
        group_scores = scores.mean(axis=1)

    elif strategy == 'least_misery':
        # Consider only positive scores; if an item has no scores, it gets 0.
        mask = scores > 0
        # For rows with at least one member score, min over non-zero entries
        group_scores = np.zeros(len(all_items), dtype=np.float32)
        has_any = mask.any(axis=1)
        # Set missing to +inf so they don't affect the min
        tmp = np.where(mask, scores, np.inf)
        group_scores[has_any] = tmp[has_any].min(axis=1)
        # Rows with no scores remain 0

    elif strategy == 'most_pleasure':
        group_scores = scores.max(axis=1)

    else:
        raise ValueError(f'Unknown strategy: {strategy}')

    out = pd.DataFrame({'item': all_items, 'score': group_scores})
    out = out.sort_values('score', ascending=False).reset_index(drop=True)
    return out


def recommend_for_group(
    engine: RecEngine,
    group_users: List[str],
    k: int = 10,
    strategy: str = 'average',
    candidate_pool_size: int = 100
) -> pd.DataFrame:
    """
    Recommend items for a group by:
      1. Getting top candidate_pool_size items for each member
      2. Aggregating with the chosen strategy
      3. Returning top-k group items, with per-member scores attached
    """
    member_frames: Dict[str, pd.DataFrame] = {}
    for u in group_users:
        recs = engine.recommend(u, n=candidate_pool_size, filter_seen_items=True)
        member_frames[u] = recs[['item', 'score']].copy() if recs is not None else pd.DataFrame(columns=['item', 'score'])

    group_df = aggregate_member_scores(member_frames, strategy=strategy)
    top = group_df.head(k).copy()

    # Also attach each member's score for inspection
    for u in group_users:
        df = member_frames[u].rename(columns={'score': f'score_{u}'})
        top = top.merge(df[['item', f'score_{u}']], on='item', how='left')

    return top.reset_index(drop=True)

print('✓ Group aggregation utilities ready.')


✓ Group aggregation utilities ready.


In [18]:
# %%
# --- Example: recommendations for a single group ---
example_users = list(train_df['user'].unique())[:5]
group = example_users[:3]
print('Example group members:', group)

group_recs = recommend_for_group(
    engine,
    group_users=group,
    k=10,
    strategy='average',
    candidate_pool_size=50
)
group_recs.head(10)


Example group members: ['1000001', '1000029', '1000045']


,item,score,score_1000001,score_1000029,score_1000045
0,361785,0.260138,NaN,0.386891,0.393523
1,363335,0.255979,NaN,0.381013,0.386925
2,351092,0.253496,0.359216,NaN,0.401271
3,361524,0.251212,0.348682,NaN,0.404955
4,362736,0.238209,0.341859,0.372767,NaN
5,341131,0.141887,NaN,NaN,0.425661
6,351204,0.141304,NaN,NaN,0.423911
7,355202,0.140796,NaN,NaN,0.422387
8,198197,0.140332,NaN,NaN,0.420997
9,343491,0.140054,NaN,0.420161,NaN


In [20]:
# %%
# --- Visual comparison: Individual vs Group performance across K ---

import matplotlib.pyplot as plt

K_VALUES = [5, 10, 20, 50, 100]
rows = []

print("Evaluating individual vs group models across K...")

for k in K_VALUES:
    ind_metrics = eval_at_k_extended(
        engine,
        test_dict,
        k=k,
        sample_size=5000,   # sample for speed
        show_progress=False
    )
    grp_metrics = eval_at_k_extended(
        group_engine,
        group_test_dict,
        k=k,
        sample_size=None,   # all groups
        show_progress=False
    )

    for model_name, metrics in [('Individual', ind_metrics), ('Group', grp_metrics)]:
        row = {'Model': model_name, 'K': k}
        row.update(metrics)
        rows.append(row)

perf_df = pd.DataFrame(rows)
print("\nPerformance summary:")
print(perf_df)

# Plot NDCG vs K
plt.figure(figsize=(8, 5))
for model in perf_df['Model'].unique():
    sub = perf_df[perf_df['Model'] == model]
    plt.plot(sub['K'], sub['NDCG@K'], marker='o', linewidth=2, label=model)

plt.xlabel('K (top-K recommendations)')
plt.ylabel('NDCG@K')
plt.title('NDCG vs K: Individual vs Group Recommender')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot HitRate vs K
plt.figure(figsize=(8, 5))
for model in perf_df['Model'].unique():
    sub = perf_df[perf_df['Model'] == model]
    plt.plot(sub['K'], sub['HitRate@K'], marker='o', linewidth=2, label=model)

plt.xlabel('K (top-K recommendations)')
plt.ylabel('HitRate@K')
plt.title('HitRate vs K: Individual vs Group Recommender')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Evaluating individual vs group models across K...


NameError: name 'group_engine' is not defined

In [ ]:
# %%
# --- Visual comparison of group aggregation strategies ---

strategies = ['average', 'least_misery', 'most_pleasure']
rows = []

print("Evaluating group aggregation strategies @K=10...")

for strat in strategies:
    tmp_group_engine = GroupRecEngineAdapter(
        base_engine=engine,
        group_members=group_members,
        strategy=strat,
        candidate_pool_size=100
    )
    metrics = eval_at_k_extended(
        tmp_group_engine,
        group_test_dict,
        k=10,
        sample_size=None,
        show_progress=False
    )
    row = {'Strategy': strat}
    row.update(metrics)
    rows.append(row)

strat_df = pd.DataFrame(rows)
print("\nGroup strategy summary (K=10):")
print(strat_df)

# Bar chart: NDCG and Recall per strategy
x = np.arange(len(strategies))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, strat_df['NDCG@K'], width, label='NDCG@10')
plt.bar(x + width/2, strat_df['Recall@K'], width, label='Recall@10')

plt.xticks(x, strat_df['Strategy'])
plt.ylabel('Score')
plt.title('Group Aggregation Strategies @K=10')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
